In [11]:
# SimpleArm Demo Notebook - Robot Arm Visualization with Environment

# ---- Cell 1: Setup ----
import sys
import numpy as np
import matplotlib.pyplot as plt
import os
import torch
import torch.nn.functional as F
import torch.nn as nn

# Add SimpleArm to path
SIMPLEARM_PATH = os.path.abspath("../external/SimpleArm/src")
sys.path.insert(0, SIMPLEARM_PATH)

print(f"SimpleArm Path: {SIMPLEARM_PATH}")
print(f"Path exists: {os.path.exists(SIMPLEARM_PATH)}")

SimpleArm Path: /Users/timomatuszewski/Desktop/SS26/Advanced_Deep_Learning_for_Robotics/Self-Supervised-Learning-for-Robot-Motion-Planning/external/SimpleArm/src
Path exists: True


In [2]:
# ---- Cell 2: Imports ----
import scipy.interpolate as interp
from simplearm.robot import RobotInfo
from simplearm.geom import Obstacles
from simplearm.viz import RobotViewer
from simplearm.costs import chomp_obstacle_cost_and_grad
from simplearm.geom import SquareGrid, SE2
from simplearm.kinematics import forward_kinematic, world_spheres_from_frames

In [3]:
# ---- Cell 3: Utility Functions ----
def query_sdf_differentiable(sdf_tensor, world_points, grid_length=2.5):
    """
    Environment differentiable encoding.
    sdf_tensor: Tensor with the distances
    world_points: Tensor with positions in meters
    grid_length: Total size
    """
    # Prepare grid_sample input: [128, 128] -> [1, 1, 128, 128]
    grid_input = sdf_tensor.unsqueeze(0).unsqueeze(0)

    # Normalize to [-1, 1]
    points_norm=world_points/(grid_length/2.0)

    # Change format to [1, N_points, 1, 2]
    grid_coords=points_norm.unsqueeze(0).unsqueeze(2)

    # Sample with bilinear interpolation
    sampled=F.grid_sample(grid_input, grid_coords, mode='bilinear',
                          padding_mode='border', align_corners=True)
    
    return sampled.reshape(-1)

def forward_kinematics_torch(q, link_lengths):
    """
    PyTorch version for the kinematics.
    q: Tensor with the configurations
    link_lenghts: Tensor or list with the link lengths
    """
    # Cumulative sum of angles
    q_cum = torch.cumsum(q, dim=-1)

    if not isinstance(link_lengths, torch.Tensor):
        link_lengths = torch.tensor(link_lengths, device=q.device, dtype=q.dtype)

    dx = torch.cos(q_cum)*link_lengths
    dy = torch.sin(q_cum)*link_lengths

    # Calculate each joint position and concatenate the origin
    x = torch.cat([torch.zeros(q.shape[0], 1, device=q.device), torch.cumsum(dx, dim=-1)], dim=-1)
    y = torch.cat([torch.zeros(q.shape[0], 1, device=q.device), torch.cumsum(dy, dim=-1)], dim=-1)

    return torch.stack((x, y), axis=-1)

def get_world_spheres_torch(q, robot_info):
    """
    Simplified version for obtaining the spheres without needing to build every frame.
    q: Tensor with the configurations
    robot_info: Contains the link lenghts and the spheres local coordinates
    """
    # Configuration
    device = q.device
    n_batch = q.shape[0]

    # Cumulate sum of angles
    q_cum = torch.cumsum(q, dim=-1)

    # Obtain each link origin
    joint_positions = forward_kinematics_torch(q, robot_info.linklengths)
    
    # Position each sphere_points
    sphere_offsets = torch.from_numpy(robot_info.spheres.xy).float().to(device)
    frame_indices = robot_info.spheres.frame_idx
    all_spheres_xy = []
    for i in range(len(frame_indices)):
        # To which link it belongs
        f_idx = frame_indices[i]

        # Link origin
        origin = joint_positions[:, f_idx, :]

        # Global angle
        angle = q_cum[:, f_idx]

        # Rotate the local offset to the global frame
        local_dist = sphere_offsets[i, 0]
        s_x = origin[:, 0] + local_dist*torch.cos(angle)
        s_y = origin[:, 1] + local_dist*torch.sin(angle)
        
        all_spheres_xy.append(torch.stack([s_x, s_y], dim=-1))
    
    return torch.stack(all_spheres_xy, dim=1)
    

In [4]:
# ---- Cell 4: Cost Functions ----
def compute_joint_limits_cost(q, q_min, q_max, weight=1.0):
    """
    Computes the robot's joint limits cost.
    q: Tensor with the configurations [N_steps, N_joints]
    q_min/q_max: Tensors with the robot's joint limits.
    """
    # If q is within the limits, the clamp result is 0 and
    # there is no penalization
    lower_violation = torch.clamp(q_min - q, min=0)
    upper_violation = torch.clamp(q - q_max, min=0)
    
    # Cuadratic penalization
    cost = torch.sum(lower_violation**2 + upper_violation**2)
    
    return weight * cost

def compute_collision_cost(distances, eps=0.1, weight=1.0):
    """
    Computes the collision cost.
    distances: Tensor with the SDF distances.
    eps: Security radio.
    """
    # Case 1: The point is inside the obstacle (negative distance)
    cost_inside = -distances + 0.5 * eps
    
    # Case 2: The point is in the danger zone2
    cost_danger = 0.5 * (distances - eps)**2 / eps
    
    # Case 3: The point is in a safe position
    cost_safe = torch.zeros_like(distances)
    
    # Combine the cases
    cost = torch.where(distances < 0, cost_inside, torch.where(
        distances <= eps, cost_danger, cost_safe))
    
    return weight * cost.sum()

def compute_smoothness_cost(q, dt=0.1, weight=1.0):
    """
    Computes the smoothness cost.
    q: Tensor with the configurations [N_steps, N_joints]
    dt: Time step
    """
    # Penalize velocity
    vel = (q[1:] - q[:-1]) / dt
    vel_cost = torch.mean(vel**2)

    # Penalize acceleration
    acc = (vel[1:] - vel[:-1]) / dt
    acc_cost = torch.mean(acc**2)

    # Combine penalizations
    total_smoothness = vel_cost + acc_cost

    return weight*total_smoothness


In [5]:
# ---- Cell 5: Create Robot Arm ----
# Create a robot arm with 3 links
linklengths = [0.5, 0.4, 0.3]
robot = RobotInfo.from_linklengths(linklengths, sphere_rad=0.08)
print("Robot arm created:")
print(robot)

Robot arm created:
A Robot with the following properties:
  Link Lengths: [0.5 0.4 0.3]
  Number of Spheres: 12
  Mass per Link: [1. 1. 1.]
  Inertia per Link: [0.1 0.1 0.1]
  Number of ignore sphere pairs: 25


In [6]:
# ---- Cell 6: Create Environment with Obstacles ----
# Define multiple obstacles in the scene
obstacles_xy = np.array([
    [0.6, 0.4],   # Obstacle 1
    [0.2, 0.8],   # Obstacle 2
    [-0.3, 0.3],  # Obstacle 3
])
obstacles_r = np.array([0.12, 0.1, 0.08])  # Radius for each obstacle

obstacles = Obstacles(
    x=obstacles_xy[:, 0],
    y=obstacles_xy[:, 1],
    r=obstacles_r
)
print(f"\nEnvironment with {len(obstacles_r)} obstacles created")


Environment with 3 obstacles created


In [ ]:
# ---- Cell 7: Visualize Robot Arm in Random Configuration ----
# Generate a random joint position
np.random.seed(42)
q_random = np.random.uniform(low=-np.pi, high=np.pi, size=(robot.n_dof,))
print(f"Random joint position: {q_random}")

# Create the viewer and visualize
viz_random = RobotViewer(q_random, robot, obstacles=obstacles)
viz_random.plot()

Random joint position: [-0.78828768  2.83192151  1.45766093]
⠙ Plotting robot... (0:00:00.09) 

In [8]:
# ---- Cell 8: Visualize Trajectory with Interpolation ----
# Generate a smooth trajectory through cubic spline interpolation
n_waypoints = 4
waypoints = np.random.uniform(low=-np.pi, high=np.pi, size=(n_waypoints, robot.n_dof))
t_waypoints = np.linspace(0, 1, n_waypoints)

# Create cubic spline and sample finely
cs = interp.CubicSpline(t_waypoints, waypoints, axis=0)
t_trajectory = np.linspace(0, 1, 50)  # 50 frames for animation
q_trajectory = cs(t_trajectory)

print(f"Trajectory with {len(q_trajectory)} configurations created")

# Visualize the trajectory
viz_traj = RobotViewer(q_trajectory, robot, obstacles=obstacles)
viz_traj.plot()

Trajectory with 50 configurations created
⠼ Plotting robot... (0:00:01.18) 

In [82]:
# ---- Cell 9: Apply Cost Functions ----

# Create the SDF
grid_length = 2.5
number_of_vox = 128
grid = SquareGrid(data=np.zeros((number_of_vox, number_of_vox)), 
    length=grid_length, origin=SE2.identity()
)
# Create circles in the grid
x = np.linspace(-grid_length/2, grid_length/2, number_of_vox)
y = np.linspace(-grid_length/2, grid_length/2, number_of_vox)
X, Y = np.meshgrid(x, y)
for i in range(len(obstacles.r)):
    dist_to_center = np.sqrt((X - obstacles.x[i])**2 + (Y - obstacles.y[i])**2)
    grid.data[dist_to_center <= obstacles.r[i]] = 1.0

sdf_data = grid.derive_sdf_from_voxels()

# Convert the SDF to a tensor
sdf_tensor = torch.from_numpy(sdf_data.data).float()

# Define joint limits
q_min = torch.tensor([-np.pi, -np.pi, -np.pi])
q_max = torch.tensor([np.pi, np.pi, np.pi])
eps_safety = 0.1  # Safety radius

# Evaluate the trajectory
q_tensor = torch.from_numpy(q_trajectory).float()

total_coll_cost = 0
total_lim_cost = 0
total_smooth_cost = 0

print("Evaluating Costs...")

# Iterate per each step
for i in range(len(q_tensor)):
    q_step = q_tensor[i].unsqueeze(0)
    
    # Joint limits cost
    total_lim_cost += compute_joint_limits_cost(q_step, q_min, q_max, weight=1.0)
    
    # Collision cost
    # Fordward kinematics
    spheres_xy = get_world_spheres_torch(q_step, robot)

    # Flatten for query_sdf
    sphere_points = spheres_xy.view(-1, 2)
    
    # Differentiable encoding
    distances = query_sdf_differentiable(sdf_tensor, sphere_points, grid_length)
        
    # Compute collision cost
    total_coll_cost += compute_collision_cost(distances, eps=eps_safety, weight=20.0)

# Smoothness cost
total_smooth_cost = compute_smoothness_cost(q_tensor, dt=0.02, weight=0.002)

print(f"--- Baseline Results ---")
print(f"Total Joint Limits Cost: {total_lim_cost.item():.4f}")
print(f"Total Collision Cost: {total_coll_cost.item():.4f}")
print(f"Total Smoothness Cost: {total_smooth_cost.item():.4f}")
print(f"Total Combined Cost: {(total_lim_cost + total_coll_cost + total_smooth_cost).item():.4f}")

Evaluating Costs...
--- Baseline Results ---
Total Joint Limits Cost: 8.5986
Total Collision Cost: 21.3940
Total Smoothness Cost: 19.7130
Total Combined Cost: 49.7057


In [83]:
# ---- Cell 10: Testing loss functions & simple GD ----

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------
# 1) Environment / robot
# ----------------------------
q_min = torch.tensor([-np.pi, -np.pi, -np.pi], device=device)
q_max = torch.tensor([ np.pi,  np.pi,  np.pi], device=device)

# ----------------------------
# 2) BAD initial pose (in obstacle)
# ----------------------------
q = torch.tensor([0.75*np.pi, -0.5*np.pi, -0.1*np.pi], device=device)
# viz = RobotViewer(q, robot, obstacles=obstacles)
# viz.plot()
q = q.clone().detach().requires_grad_(True)


# ----------------------------
# 3) Optimizer
# ----------------------------
opt = torch.optim.SGD([q], lr=0.005)

# ----------------------------
# 4) storage
# ----------------------------
trajectory = [q.detach().cpu().numpy().copy()]

# ----------------------------
# 5) iterative improvement
# ----------------------------
it = 0
losss = 1000
while losss > 0.01 and it < 100:

    opt.zero_grad()

    q_step = q.unsqueeze(0)

    # ---- collision cost ----
    spheres = get_world_spheres_torch(q_step, robot)
    pts = spheres.view(-1, 2)

    dist = query_sdf_differentiable(
        sdf_tensor,
        pts
    )

    loss = compute_collision_cost(dist, eps=0.1, weight=20.0)
    loss = loss + compute_joint_limits_cost(q_step, q_min, q_max)
    losss = loss.item()

    loss.backward()
    opt.step()

    # optional clamp
    with torch.no_grad():
        q[:] = torch.clamp(q, -np.pi, np.pi)

    # store snapshot (IMPORTANT)
    trajectory.append(q.detach().cpu().numpy().copy())
    it += 1

    if it % 10 == 0:
        print(f"iter {it} | loss {loss.item():.4f} | q: {q.detach().cpu().numpy()}")

q_traj = np.stack(trajectory)  # [100, dof]
print(q_traj.shape)
viz = RobotViewer(q_traj, robot, obstacles=obstacles)
viz.plot()


iter 10 | loss 0.1324 | q: [ 2.0195937  -1.7129064  -0.39319706]
iter 20 | loss 0.0237 | q: [ 1.9848163  -1.658623   -0.37381488]
(26, 3)
⠙ Plotting robot... (0:00:00.35) 

In [84]:
# ---- Cell 11: Network Architecture ----


def build_bspline_matrix(T, C, degree=3, device="cpu"):
    """
    Builds a B-spline basis matrix A ∈ [T, C]
    Each row sums weighted cubic B-spline basis functions.
    """

    def cox_de_boor(t, i, k, knots):
        if k == 0:
            return ((knots[i] <= t) & (t < knots[i+1])).float()
        denom1 = knots[i+k] - knots[i]
        denom2 = knots[i+k+1] - knots[i+1]

        term1 = 0 if denom1 == 0 else (t - knots[i]) / denom1 * cox_de_boor(t, i, k-1, knots)
        term2 = 0 if denom2 == 0 else (knots[i+k+1] - t) / denom2 * cox_de_boor(t, i+1, k-1, knots)

        return term1 + term2

    # time grid
    t_vals = torch.linspace(0, 1, T, device=device)

    # clamped uniform knots
    knots = torch.linspace(0, 1, C - degree + 1, device=device)
    knots = torch.cat([
        torch.zeros(degree, device=device),
        knots,
        torch.ones(degree, device=device)
    ])

    A = torch.zeros(T, C, device=device)

    for ti, t in enumerate(t_vals):
        for i in range(C):
            A[ti, i] = cox_de_boor(t, i, degree, knots)

    return A


class EnvEncoder(nn.Module):
    def __init__(self, latent=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 5, 2, 2),
            nn.ReLU(),
            nn.Conv2d(16, 32, 5, 2, 2),
            nn.ReLU(),
            nn.Conv2d(32, 64, 5, 2, 2),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.fc = nn.Linear(64, latent)

    def forward(self, sdf):
        x = self.net(sdf).squeeze(-1).squeeze(-1)
        return self.fc(x)

class StateEncoder(nn.Module):
    def __init__(self, dof=3, hidden=128):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(2 * dof, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU()
        )

    def forward(self, q_start, q_goal):
        x = torch.cat([q_start, q_goal], dim=-1)
        return self.net(x)

class ControlPointDecoder(nn.Module):
    def __init__(self, latent_env=64, latent_state=128, C=10, dof=3):
        super().__init__()

        self.C = C
        self.dof = dof

        self.mlp = nn.Sequential(
            nn.Linear(latent_env + latent_state, 256),
            nn.ReLU(),
            nn.Linear(256, C * dof)
        )

    def forward(self, z_env, z_state):
        z = torch.cat([z_env, z_state], dim=-1)
        out = self.mlp(z)
        return out.view(-1, self.C, self.dof)


# ----------------------------
# Full Model: Warm Start Planner (B-spline + delta)
# ----------------------------
class WarmStartPlanner(nn.Module):
    def __init__(self, dof=3, T=50, C=10):
        super().__init__()

        self.T = T
        self.C = C
        self.dof = dof

        self.env_encoder = EnvEncoder()
        self.state_encoder = StateEncoder(dof=dof)
        self.decoder = ControlPointDecoder(dof=dof, C=C)

    def forward(self, q_start, q_goal, sdf):
        device = q_start.device

        # Encode inputs
        z_env = self.env_encoder(sdf)
        z_state = self.state_encoder(q_start, q_goal)

        # Predict delta control points
        delta_ctrl = self.decoder(z_env, z_state)

        # Baseline: straight line control points
        B = torch.linspace(0, 1, self.C, device=device).unsqueeze(-1)
        baseline = q_start.unsqueeze(1) * (1 - B) + q_goal.unsqueeze(1) * B

        # Combine baseline + delta
        ctrl_pts = baseline + delta_ctrl

        A = build_bspline_matrix(self.T, self.C, degree=3, device=device)
        traj = torch.einsum("tc,bcd->btd", A, ctrl_pts)

        return traj

In [85]:
# ---- Cell 12: Instantiate model & test model ----

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = WarmStartPlanner(dof=3, T=50).to(device)
model.eval()

# start / goal
q_start = torch.tensor([[0.0, 0.5, -0.3]], dtype=torch.float32).to(device)
q_goal  = torch.tensor([[1.0, -0.5, 0.8]], dtype=torch.float32).to(device)

# SDF
sdf_np = sdf_tensor.numpy()
sdf = torch.from_numpy(sdf_np).float().unsqueeze(0).unsqueeze(0).to(device)

with torch.no_grad():
    q_traj = model(q_start, q_goal, sdf)   # <-- THIS is final trajectory

# ---------------------------
# visualize
# ---------------------------
print("Trajectory shape:", q_traj.shape)

q_traj_np = q_traj.squeeze(0).cpu().numpy()

viz = RobotViewer(q_traj_np, robot, obstacles=obstacles)
viz.plot()

Trajectory shape: torch.Size([1, 50, 3])
⠹ Plotting robot... (0:00:01.16) 